# SPGER0046 Recurso solar e modelagem fotovoltaica

<details>
<summary>Descrição e método</summary>

**Versão 1.2.2.** Farm Solar Array PVDAQ 2107, Califórnia, abril a junho de 2024. Entradas: POA, temperatura do ar e vento locais. Saída: geração AC calculada, comparada com o medidor. NASA POWER não é usado na cadeia principal. As subseções relacionam fundamentos, dados, cálculos e resultados. As descrições usam HTML details fechado por padrão; a renderização e o recolhimento por título variam entre JupyterLab e Colab. O relatório HTML exportado conserva os blocos recolhíveis. A exportação ao Drive requer autenticação pessoal no Colab.

</details>

**Como usar:** execute as células na ordem. As saídas ficam em outputs, separadas em tabelas, quadros, graficos, fluxogramas e figuras. Cada figura final possui PDF vetorial e PNG a 600 dpi nos temas claro e escuro em secoes. Quatro DOCX separados reúnem as tabelas e quadros de cada seção; as coleções consolidadas servem para consulta. A exportação pessoal ao Drive é opcional e exige autenticação no Colab. **Inserção editorial única:** cada figura, gráfico e tabela possui somente uma subseção responsável. Usar o mapa config/distribuicao_editorial.json e citar o ID em outras partes, sem reinserir o material. As quatro versões de uma ilustração são alternativas de formato e tema.

**Inventário para redação:** [Word geral e quadros de equações por seção](https://github.com/samuelasroma-byte/SPGER0046_Energias_Renovaveis/blob/main/docs/equacoes/README.md). Cada relação possui ID, função e local de inserção; o inventário distingue o relações do notebook dos complementos do código.

**Sumário:** [1.1](#sec_1_1) · [1.2](#sec_1_2) · [1.3](#sec_1_3) · [1.4](#sec_1_4) · [2.1](#sec_2_1) · [2.2](#sec_2_2) · [2.3](#sec_2_3) · [2.4](#sec_2_4) · [3.1](#sec_3_1) · [3.2](#sec_3_2) · [3.3](#sec_3_3) · [3.4](#sec_3_4) · [3.5](#sec_3_5) · [3.6](#sec_3_6) · [4.1](#sec_4_1) · [4.2](#sec_4_2) · [4.3](#sec_4_3) · [4.4](#sec_4_4) · [4.5](#sec_4_5)

## 0 Preparação e sistema de referência

<details>
<summary>Descrição e método</summary>

O modelo representa uma instalação real. Equipamentos: 2.880 módulos e 24 inversores. P₀ de normalização=893 kWp; soma de placa=892,8 kWp; soma STC CEC=891,648 kWp. O equivalente 20×6 não é cabeamento verificado. Nenhuma lacuna diurna é interpolada. Os arquivos oficiais são baixados apenas se o ZIP local estiver ausente, com conferência SHA-256.

</details>

In [ ]:
from pathlib import Path
import sys,subprocess,importlib.util,importlib.metadata,os,json,base64,io,zipfile,hashlib
if importlib.util.find_spec("packaging") is None:
    subprocess.run([sys.executable,"-m","pip","install","packaging"],check=True)
from packaging.requirements import Requirement
requirements={"numpy":"numpy>=2,<3","pandas":"pandas>=2.2,<3","scipy":"scipy>=1.13,<2",
    "matplotlib":"matplotlib>=3.9,<4","pvlib":"pvlib==0.16.1","python-docx":"python-docx>=1.1,<2","Pillow":"Pillow>=10,<13"}
missing=[]
for package,req in requirements.items():
    try:
        if importlib.metadata.version(package) not in Requirement(req).specifier:missing.append(req)
    except importlib.metadata.PackageNotFoundError:missing.append(req)
if missing:subprocess.run([sys.executable,"-m","pip","install",*missing],check=True)
support=base64.b64decode("...")
assert hashlib.sha256(support).hexdigest()=="f9ceaa991723a4ba683285953152d35197f11d3b3bbfc44b25a69d488e6068de"
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/"src/projeto_fv.py").exists()),None)
if ROOT is None:ROOT=Path.cwd()/"Projeto_FV_PVDAQ_2107_2024"
update=not (ROOT/"config/projeto.json").exists()
if not update:update=json.loads((ROOT/"config/projeto.json").read_text()).get("version")!="1.2.2"
if not (ROOT/"config/distribuicao_editorial.json").exists():update=True
if update:
    with zipfile.ZipFile(io.BytesIO(support)) as archive:
        for item in archive.infolist():
            dest=(ROOT/item.filename).resolve()
            assert dest.is_relative_to(ROOT.resolve())
            if dest.exists():
                backup=ROOT/"backup_antes_v1_2"/item.filename
                backup.parent.mkdir(parents=True,exist_ok=True)
                if not backup.exists():backup.write_bytes(dest.read_bytes())
            dest.parent.mkdir(parents=True,exist_ok=True)
            dest.write_bytes(archive.read(item.filename))
sys.path.insert(0,str(ROOT/"src"))
import projeto_fv as fv,refinamento_fv as rf,relatorio_fv as rel,editorial_fv as editorial
import importlib
for module in [fv,rf,rel,editorial]:importlib.reload(module)
import numpy as np,pandas as pd,pvlib
from IPython.display import display,Markdown,HTML
display(HTML("<style>.text_cell_render,.jp-RenderedHTMLCommon{font:20px/1.65 Arial,sans-serif}.dataframe{font-size:17px}td,th{padding:8px}summary{cursor:pointer}</style>"))
CFG=fv.load_config(ROOT)
EXECUTAR_INVERSAO_EXPLORATORIA=True
GERAR_DOCX=True
REUTILIZAR_FIGURAS_VERIFICADAS=True
TEMA_VISUALIZACAO="escuro"
SALVAR_RESULTADOS_NO_DRIVE=False
PASTA_RESULTADOS_ID="ID_RESTRITO_DA_EQUIPE"
print("Versão",CFG["version"],"| NASA POWER fora da cadeia principal")
display(pd.DataFrame([{"biblioteca":k,"versão":importlib.metadata.version(k)} for k in requirements]))


### 0.1 Auditoria e cálculo da cadeia principal

<details>
<summary>Descrição e método</summary>

O filtro diurno utiliza elevação NOAA ≥10°; SPA calcula a geometria no meio dos intervalos. Carimbos são tratados como início de intervalos locais America/Los_Angeles, interpretando PST8PDT dos metadados. Posição representativa em +7,5 minutos para as médias de 15 minutos e +30 minutos para a inversão horária; convenção temporal documentada, sem otimização por defasagem da geração. As médias de 15 minutos dos canais de 5 minutos exigem três amostras. O fator empírico secundário usa somente abril.

</details>

In [ ]:
CTX=fv.complete_pipeline(ROOT,make_plots=False,include_nasa=False,reverse=EXECUTAR_INVERSAO_EXPLORATORIA)
D,A=CTX["data"],CTX["model_a"]
print("Cobertura conjunta diurna:",100*CTX["audit"]["available_daylight_15m"]/CTX["audit"]["expected_daylight_15m"],"%")
print("O quadro de disponibilidade dos dados é exibido exclusivamente em 1.4.")


### 0.2 Exportações e diagnósticos

<details>
<summary>Descrição e método</summary>

Dados e filtros idênticos geram os dois temas. Cada figura possui fonte, legenda, natureza da evidência e subseção no catálogo. Cada ilustração é exibida somente na sua subseção de inserção editorial. As demais subseções a citam sem repetir a exibição.

</details>

In [ ]:
rf.generate_all(CTX,reuse_checked=REUTILIZAR_FIGURAS_VERIFICADAS)
rel.coverage(CTX)
print("Figuras:",len(CTX["figures"]),"| arquivos finais PDF/PNG:",4*len(CTX["figures"]))
print("Saídas:",ROOT/"outputs")


# 1 Fundamentos e caracterização do recurso solar

<details>
<summary>Descrição e método</summary>

Caracterizar e representar o recurso solar. **Responsável:** Samuel.

</details>

## 1.1 Geometria solar e irradiância extraterrestre <a id="sec_1_1"></a>

<details>
<summary>Descrição e método</summary>

Calcular a posição solar com SPA e a irradiância extraterrestre com Spencer. O azimute é contado a partir do norte no sentido horário.

**Entradas:** Latitude, longitude, altitude e horário dos intervalos.

$$\cos\theta_z=\sin\varphi\sin\delta+\cos\varphi\cos\delta\cos\omega,\qquad G_{0h}=G_{0n}\max(\cos\theta_z,0).$$

**Evidência:** Cálculo físico com coordenadas documentadas. **Limite:** Não há observação independente de ângulo solar; a geometria pode ser calculada.

**Saídas:** Geometria solar e quadro de convenções. **Referências:** R03, R04, R05.

</details>

In [ ]:
display(D[["zenith","azimuth","aoi_deg","dni_extra_Wm2","extraterrestrial_horizontal_Wm2"]].iloc[40:45].round(3))
rf.show(CTX,['G01_recurso_geometria'],TEMA_VISUALIZACAO)

## 1.2 Grandezas radiométricas e componentes da radiação <a id="sec_1_2"></a>

<details>
<summary>Descrição e método</summary>

Irradiância é potência por área; irradiação é sua integral no tempo. GHI é horizontal, DNI é normal aos raios solares e DHI é difusa horizontal. POA se refere ao plano do arranjo.

**Entradas:** POA local medida; componentes reconstruídas quando a inversão estiver ativa.

$$GHI=DNI\max(\cos\theta_z,0)+DHI,\qquad H=\sum_i G_i\Delta t_i.$$

**Evidência:** Medição POA e estimativa exploratória das componentes. **Limite:** Sem GHI/DNI/DHI locais medidos; reconstrução não equivale a observação.

**Saídas:** Quadro das grandezas e componentes estimadas. **Referências:** R01, R14, R18, R29.

</details>

In [ ]:
print("POA é medida. GHI/DNI/DHI disponíveis são estimativas da inversão.")
rf.show(CTX,['F02_componentes_POA'],TEMA_VISUALIZACAO)

## 1.3 Modulação atmosférica e variabilidade espaço temporal <a id="sec_1_3"></a>

<details>
<summary>Descrição e método</summary>

Nuvens, aerossóis e vapor d’água modificam o recurso solar. A série permite descrever a variabilidade intradiária e entre dias; a seleção dos dias ilustrativos usa apenas POA e cobertura.

**Entradas:** Série de POA, intervalo de integração e critérios de cobertura.

$$V_{\Delta G}=\frac{\operatorname{média}(|G_i-G_{i-1}|)}{\operatorname{média}(G_i)}.$$

**Evidência:** Descrição observacional de variabilidade. **Limite:** Não se atribuem causas atmosféricas isoladas nem climatologia anual a um trimestre.

**Saídas:** Critério dos dias ilustrativos e irradiação pareada. **Referências:** R01, R25.

</details>

In [ ]:
display(CTX["analytics_tables"]["criterio_dias_ilustrativos"].iloc[[0,-1]])
rf.show(CTX,['G02_variabilidade_recurso'],TEMA_VISUALIZACAO)

## 1.4 Medição e estimativa do recurso solar <a id="sec_1_4"></a>

<details>
<summary>Descrição e método</summary>

Inspecionar integridade, unidades, lacunas, simultaneidade e limites físicos. Medições de superfície, satélites, reanálises e atlas têm representatividades diferentes; os produtos em grade são contextualizados, sem uso na cadeia principal.

**Entradas:** Arquivos PVDAQ, metadados, canais medidos e resoluções de 5/15 minutos.

$$C=100\,N_{\mathrm{pares}}/N_{\mathrm{esperados}}.$$

**Evidência:** Auditoria de disponibilidade e documentação. **Limite:** Disponibilidade elevada não certifica sensores. Altura do anemômetro e alguns metadados permanecem incompletos.

**Saídas:** Situação dos dados, proveniência e cobertura. **Referências:** R01, R02, R21, R22, R23, R25, R26.

</details>

In [ ]:
display(pd.DataFrame([CTX["audit"]])); display(fv.data_status())
rf.show(CTX,['FL01_fluxo_projeto'],TEMA_VISUALIZACAO)

# 2 Condições ambientais e de entorno da conversão fotovoltaica

<details>
<summary>Descrição e método</summary>

Relacionar ambiente e instalação à irradiância e à temperatura. **Responsável:** Vannessa e Thais.

</details>

## 2.1 Condições térmicas <a id="sec_2_1"></a>

<details>
<summary>Descrição e método</summary>

Temperatura ambiente, vento e trocas de calor condicionam a operação do módulo. As distribuições observadas e o cenário de menor ventilação ilustram a passagem do ambiente à modelagem térmica da seção 3.2.

**Entradas:** Temperatura do ar e vento locais; POA medida.

$$T_{\mathrm{op}}=T_a+\frac{G_{\mathrm{ef}}}{U_0+U_1v}.$$

**Evidência:** Medições ambientais e temperatura estimada. **Limite:** U₀ e U₁ genéricos; T operacional é aproximação de célula, sem validação térmica independente.

**Saídas:** Estatística ambiental e sensibilidade de ventilação. **Referências:** R07.

</details>

In [ ]:
display(CTX["analytics_tables"]["estatistica_descritiva"].iloc[:4])
rf.show(CTX,['G03_condicoes_ambientais'],TEMA_VISUALIZACAO)

## 2.2 Albedo e ambiente radiativo da superfície <a id="sec_2_2"></a>

<details>
<summary>Descrição e método</summary>

O albedo expressa refletância do entorno. No exercício de transposição, varia-se 0,10/0,20/0,40 mantendo as componentes reconstruídas na referência 0,20. O arranjo real é monofacial; bifacialidade exige caracterização traseira adicional.

**Entradas:** Albedo assumido e componentes estimadas da inversão.

$$G_{\mathrm{solo}}=\rho_g\,GHI\,\frac{1-\cos\beta}{2}.$$

**Evidência:** Cenário de sensibilidade. **Limite:** Expressão isotrópica para reflexão do solo. Não há albedo medido; a leitura POA principal não recebe uma segunda contribuição refletida.

**Saídas:** Sensibilidade de albedo e quadro de hipóteses. **Referências:** R06, R14, R29.

</details>

In [ ]:
display(CTX["environment_scenarios"]) if len(CTX["environment_scenarios"]) else print("Albedo: cenário omitido sem inversão.")
rf.show(CTX,['G22_albedo'],TEMA_VISUALIZACAO)

## 2.3 Sombreamento sujidade e obstruções locais <a id="sec_2_3"></a>

<details>
<summary>Descrição e método</summary>

A sujeira pode atenuar a luz incidente e diferir entre módulos e sensor POA. Os cenários são hipóteses de atenuação adicional à leitura. Sombreamento parcial pode envolver mismatch e diodos de bypass, que um redutor uniforme não reproduz.

**Entradas:** Cenários de atenuação adicional de 0%, 2% e 5%; alertas operacionais.

$$G_{\mathrm{ef}}=(1-s)G_{\mathrm{POA}}.$$

**Evidência:** Sensibilidade hipotética e fundamentos. **Limite:** Sem histórico de limpeza e mapa de sombras; o resíduo não identifica a perda real de sujidade ou sombra.

**Saídas:** Sensibilidade e alertas sem atribuição causal automática. **Referências:** R28, R25.

</details>

In [ ]:
display(CTX["environment_sensitivity"])
rf.show(CTX,['G20_sujidade_ventilacao'],TEMA_VISUALIZACAO)

## 2.4 Geometria de instalação e exposição do arranjo fotovoltaico <a id="sec_2_4"></a>

<details>
<summary>Descrição e método</summary>

A instalação define exposição e incidência. A inclinação é medida em relação à horizontal. Espaçamento entre fileiras e exposição traseira precisam de geometria adicional; em um exemplo de sombra coplanar, uma altura h gera projeção horizontal h/tan(altura solar).

**Entradas:** Inclinação de 25° e azimute de 180° documentados; fachadas hipotéticas.

$$\cos\theta_i=\cos\theta_z\cos\beta+\sin\theta_z\sin\beta\cos(\gamma_s-\gamma_a),\quad \ell=h/\tan\alpha_s.$$

**Evidência:** Geometria documentada e extensão didática. **Limite:** A expressão de projeção é um exemplo plano sem dimensão de fileira real; não calcula o sombreamento específico da usina.

**Saídas:** Convenções geométricas e limite de dados de espaçamento. **Referências:** R03, R09, R29.

</details>

In [ ]:
display(pd.DataFrame([CFG["site"]]))
rf.show(CTX,['F01_plano_angulos'],TEMA_VISUALIZACAO)

# 3 Modelagem da conversão fotovoltaica

<details>
<summary>Descrição e método</summary>

Transformar as entradas em temperatura, potência e energia. **Responsável:** Gabriela e Giulia.

</details>

## 3.1 Transposição para o plano do arranjo <a id="sec_3_1"></a>

<details>
<summary>Descrição e método</summary>

A cadeia usual transpõe GHI/DNI/DHI para POA. Como a instalação já mede POA, o modelo principal começa dessa observação. A inversão opcional usa Driesse 2024, Erbs–Driesse e Perez–Driesse para exercícios de orientação/albedo.

**Entradas:** Componentes reconstruídas, posição solar e geometria.

$$G_{\mathrm{POA}}=DNI\max(\cos\theta_i,0)+G_{\mathrm{céu}}+G_{\mathrm{solo}}.$$

**Evidência:** Medição direta e inversão exploratória. **Limite:** Há possibilidade de soluções não únicas; o fechamento de POA é teste numérico, sem validação independente das componentes.

**Saídas:** Diagnóstico horário da inversão e critérios de exclusão. **Referências:** R06, R14, R15, R16, R18.

</details>

In [ ]:
display(pd.DataFrame([CTX.get("reverse_summary",{"executed":False})]))
rf.show(CTX,['G23_componentes_estimadas'],TEMA_VISUALIZACAO)

## 3.2 Modelagem da temperatura operacional <a id="sec_3_2"></a>

<details>
<summary>Descrição e método</summary>

A temperatura estimada entra no modelo elétrico. U₀=25 W/(m² K) e U₁=6,84 W s/(m³ K) são parâmetros genéricos de referência; a sensibilidade de ventilação usa 15 e 3, respectivamente.

**Entradas:** POA, temperatura do ar, vento e coeficientes Faiman.

$$T_{\mathrm{op}}=T_a+G_{\mathrm{ef}}/(U_0+U_1v).$$

**Evidência:** Modelo térmico. **Limite:** Ausência de temperatura de módulo/célula e altura de vento suficientemente documentada.

**Saídas:** Temperatura estimada e registro dos coeficientes. **Referências:** R07.

</details>

In [ ]:
display(pd.DataFrame([{k:CFG["assumptions"][k] for k in ["faiman_u0","faiman_u1","faiman_temperature_used_as_cell_proxy"]}]))
rf.show(CTX,['G04_temperatura_estimada'],TEMA_VISUALIZACAO)

## 3.3 Modelagem elétrica do módulo fotovoltaico <a id="sec_3_3"></a>

<details>
<summary>Descrição e método</summary>

O modelo de diodo único fornece curvas I–V, P–V e o ponto de máxima potência. Compare irradiâncias a 25 °C e temperaturas a 1.000 W/m². Registre divergências de classificação e parâmetros entre CEC e metadados.

**Entradas:** Parâmetros CEC, irradiância efetiva e temperatura estimada.

$$I=I_L-I_0[\exp((V+IR_s)/a)-1]-(V+IR_s)/R_{sh},\quad P=VI.$$

**Evidência:** Modelo elétrico e curvas calculadas. **Limite:** Sem varreduras I–V experimentais da instalação; as curvas não constituem validação do módulo.

**Saídas:** Curvas calculadas e identificação de MPP. **Referências:** R08, R09, R10.

</details>

In [ ]:
print("Curvas CEC calculadas; não são varreduras experimentais.")
rf.show(CTX,['G05_curvas_irradiancia', 'G06_curvas_temperatura'],TEMA_VISUALIZACAO)

## 3.4 Modelagem do arranjo fotovoltaico em corrente contínua <a id="sec_3_4"></a>

<details>
<summary>Descrição e método</summary>

Em uma associação ideal, as tensões somam em série e as correntes em paralelo. O exemplo 20 módulos × 6 strings por grupo corresponde a 120 módulos, mas não confirma o cabeamento real ou a ligação aos MPPTs.

**Entradas:** MPP do módulo, 2.880 módulos, 24 grupos/inversores e perdas DC assumidas.

$$V_a=N_sV_m,\quad I_a=N_pI_m,\quad P_a=N_sN_pP_m,\quad P_{\mathrm{DC}}=P_a\prod_j(1-\ell_j).$$

**Evidência:** Modelo agregado e equivalente didático. **Limite:** Strings/MPPTs reais incompletos; V×I disponível não foi adotado como potência DC observada certificada.

**Saídas:** Tensão, corrente e potência do equivalente em CSV. **Referências:** R08, R10, R13.

</details>

In [ ]:
display(CTX["analytics_tables"]["arranjo_equivalente_diurno"].head().round(3))
rf.show(CTX,['F03_arranjo_eletrico'],TEMA_VISUALIZACAO)

## 3.5 Conversão de CC para CA e modelagem do inversor <a id="sec_3_5"></a>

<details>
<summary>Descrição e método</summary>

A curva PVWatts estima conversão e clipping. O limite de referência é 30 kW por inversor; 27,6 kW permanece como sensibilidade documentada. Verificar a faixa real de MPPT exige topologia e configuração de campo adicionais.

**Entradas:** Potência DC calculada, eficiência e limites assumidos do inversor.

$$P_{\mathrm{AC}}=\min[\eta(P_{\mathrm{DC}})P_{\mathrm{DC}},P_{\mathrm{AC,max}}],\quad P_{\mathrm{dc0,inv}}=P_{\mathrm{AC,max}}/\eta_{\mathrm{nom}}.$$

**Evidência:** Modelo de conversão e sensibilidade. **Limite:** A simulação agregada não verifica todas as restrições reais de tensão/MPPT nem confirma o limite instalado.

**Saídas:** Curva de eficiência, clipping calculado e sensibilidade AC. **Referências:** R11, R13.

</details>

In [ ]:
display(CTX["capacity_sensitivity"].round(4))
rf.show(CTX,['G07_inversor'],TEMA_VISUALIZACAO)

## 3.6 Integração temporal e estimativa de energia <a id="sec_3_6"></a>

<details>
<summary>Descrição e método</summary>

Integre potência média por duração de cada intervalo. Compare séries e energias nas mesmas janelas. Apresente energia mensal e trimestral observada completa separadamente dos intervalos diurnos pareados.

**Entradas:** Potência média em intervalos de 15 minutos e máscaras temporais.

$$E=\sum_i P_i\Delta t_i,\qquad \Delta t=0,25\ \mathrm{h}.$$

**Evidência:** Integração calculada e geração observada. **Limite:** O recorte é trimestral, não uma medição anual; energia pareada não equivale ao total integral do dia.

**Saídas:** Energia diária/mensal, cobertura e totais integrais separados. **Referências:** R12.

</details>

In [ ]:
display(CTX["analytics_tables"]["energia_e_indicadores_mensais"].round(3))
rf.show(CTX,['G08_series_potencia', 'G09_energia_diaria', 'G10_energia_mensal'],TEMA_VISUALIZACAO)

# 4 Avaliação de desempenho perdas e validação

<details>
<summary>Descrição e método</summary>

Quantificar desempenho e confrontar a geração calculada com observações. **Responsável:** Júlia Ribeiro e Lucas Simão.

</details>

## 4.1 Balanço energético e cadeia de perdas <a id="sec_4_1"></a>

<details>
<summary>Descrição e método</summary>

Integre a cadeia óptica, térmica, elétrica DC, conversão e limite AC. A diferença entre estágios pertence ao modelo; um desvio do medidor pode envolver fenômenos e incertezas não identificados.

**Entradas:** Estágios do modelo e alertas de potência sob irradiância elevada.

$$\Delta E_j=E_j-E_{j+1},\qquad \sum_j\Delta E_j=E_1-E_{\mathrm{final}}.$$

**Evidência:** Decomposição calculada. **Limite:** Não se identificam perdas ópticas/operacionais reais isoladas sem medições adicionais; ganhos sob frio podem ocorrer.

**Saídas:** Energia por estágio e alertas operacionais. **Referências:** R08, R11, R12, R25.

</details>

In [ ]:
display(CTX["energy_stages"].round(3)); display(CTX["operational_flags"].head())
rf.show(CTX,['G11_cadeia_energetica'],TEMA_VISUALIZACAO)

## 4.2 Produtividades de referência do arranjo e final <a id="sec_4_2"></a>

<details>
<summary>Descrição e método</summary>

Yr é o recurso em horas equivalentes; Ya representa a energia DC por capacidade; Yf representa energia AC por capacidade. HSP/PSH corresponde ao mesmo conceito de recurso quando plano e janela são os mesmos.

**Entradas:** Irradiação POA, energia DC/AC e potência declarada de 893 kWp.

$$Y_r=H_{\mathrm{POA}}/G_{\mathrm{ref}},\quad Y_a=E_{\mathrm{DC}}/P_0,\quad Y_f=E_{\mathrm{AC}}/P_0,\quad G_{\mathrm{ref}}=1\ \mathrm{kW/m^2}.$$

**Evidência:** Indicadores sobre observações e modelo. **Limite:** Ya é modelada; HSP da janela pareada não é duração de brilho solar nem total integral diário.

**Saídas:** Yr, Ya, Yf e HSP na janela definida. **Referências:** R12, R30.

</details>

In [ ]:
display(CTX["analytics_tables"]["energia_e_HSP_diurna_pareada"][["day_local","PSH_POA_paired_h","paired_intervals"]].head().round(4))
rf.show(CTX,['G21_HSP_POA_pareada'],TEMA_VISUALIZACAO)

## 4.3 Razão de desempenho e decomposição normalizada das perdas <a id="sec_4_3"></a>

<details>
<summary>Descrição e método</summary>

Compare PR observado e modelado com irradiação e energia da mesma janela. Lc e Ls decompõem o modelo em perdas de captura e sistema. Informe a normalização e o papel da potência declarada.

**Entradas:** Produtividades calculadas na mesma máscara temporal.

$$PR=Y_f/Y_r,\quad L_c=Y_r-Y_a,\quad L_s=Y_a-Y_f.$$

**Evidência:** PR observado pareado e perdas calculadas. **Limite:** Não combinar energia integral do trimestre com POA parcial; perdas calculadas não são identificação experimental de cada causa.

**Saídas:** PR, Lc, Ls e janela de integração. **Referências:** R12, R30.

</details>

In [ ]:
display(CTX["yields"].round(5))
rf.show(CTX,['G12_produtividades_PR'],TEMA_VISUALIZACAO)

## 4.4 Validação de modelos de geração fotovoltaica <a id="sec_4_4"></a>

<details>
<summary>Descrição e método</summary>

Avalie viés, magnitude do erro, correlação e determinação. Erro=simulado−observado. MBE, MAE e RMSE são em kW; nMBE/nRMSE são divididos pela média observada dos respectivos pares. O modelo físico é separado do fator empírico ajustado somente em abril.

**Entradas:** Mesmos pares de potência AC medida e simulada.

$$MBE=\bar e,\quad MAE=\overline{|e|},\quad RMSE=\sqrt{\overline{e^2}},\quad R^2=1-\frac{\sum e_i^2}{\sum(y_i-\bar y)^2}.$$

**Evidência:** Comparação observacional da saída AC. **Limite:** Junho é teste temporal separado, não instalação independente. Correlação elevada não prova exatidão; nenhum limiar universal de aceitação foi adotado.

**Saídas:** Métricas gerais, por mês/hora/POA, resíduos e consistência de medidores. **Referências:** R27, R25.

</details>

In [ ]:
display(CTX["metrics"].round(5)); display(CTX["analytics_tables"]["metricas_por_mes"].round(4)); display(pd.DataFrame([CTX["meter_inverter_consistency"]]).round(4))
rf.show(CTX,['G13_validacao', 'G14_residuos_temporais', 'G15_residuos_potencia', 'G16_distribuicao_residuos', 'G17_metricas_mensais', 'G18_vies_horario', 'G19_consistencia_medicoes'],TEMA_VISUALIZACAO)

## 4.5 Avaliação de desempenho de sistemas integrados à edificação <a id="sec_4_5"></a>

<details>
<summary>Descrição e método</summary>

Explore fachadas sul/leste/oeste e menor ventilação. O exemplo separado de 100 m² semitransparentes usa transmissão solar 0,25, reflexão 0,12, absorção 0,63 e transmissão visível 0,35. As propriedades não caracterizam o módulo real.

**Entradas:** Componentes reconstruídas, meteorologia local e coeficientes BIPV assumidos.

$$P_e=\eta GA,\quad Q_{\mathrm{abs}}=(\alpha-\eta)GA,\quad Q_{\mathrm{in}}=\tau_sGA+f_{\mathrm{in}}Q_{\mathrm{abs}},\quad SHGC_{\mathrm{eq}}=\tau_s+f_{\mathrm{in}}(\alpha-\eta).$$

**Evidência:** Cenário geométrico e balanço hipotético. **Limite:** Somente horas elegíveis da inversão; sem fachada medida, SHGC certificado, cálculo de lux ou balanço térmico completo do edifício.

**Saídas:** Produtividade/PR de cenários e conservação da partição solar. **Referências:** R14, R19, R20.

</details>

In [ ]:
display(CTX["bipv_comparison"].round(4)) if "bipv_comparison" in CTX else print("BIPV omitido sem inversão.")
rf.show(CTX,['G24_BIPV_orientacoes', 'G25_BIPV_balanco', 'F04_geometrias_BIPV'],TEMA_VISUALIZACAO)

# 5 Referências

<details>
<summary>Descrição e método</summary>

Inventário das fontes dos dados, modelos e limites. O levantamento teórico por tópico está no notebook bibliografia/02_Fundamentacao_Teorica_por_Subsecao.ipynb e em bibliografia/outputs/. Não substitui leitura crítica e redação final. Fontes NASA pertencem ao histórico/comparação de produtos, sem entrada na cadeia principal.

</details>

In [ ]:
REFERENCIAS=json.loads((ROOT/"references/referencias_projeto.json").read_text())
for r in REFERENCIAS:
    display(Markdown("**"+r["id"]+"** — "+r["citation"]+"\n\n"+r["url"]+"\n\nUso: "+r["role"]))


# 6 Documentos e relatório

<details>
<summary>Descrição e método</summary>

Quatro DOCX editoriais contêm apenas as tabelas e quadros atribuídos a cada seção. As seis coleções consolidadas e o dossiê servem como consulta da execução. Papel A4, margens superior/esquerda de 3 cm e inferior/direita de 2 cm; Times New Roman 12 no texto nativo, com títulos de tabelas e figuras centralizados. O relatório HTML possui tema alternável e descrições recolhíveis. Os documentos Word e PDF apresentam os mesmos resultados e convenções.

</details>

In [ ]:
if GERAR_DOCX:
    python_documentos=os.environ.get("CODEX_PRIMARY_RUNTIME_PYTHON",sys.executable)
    subprocess.run([python_documentos,str(ROOT/"scripts/gerar_documentos.py")],check=True)
    subprocess.run([python_documentos,str(ROOT/"scripts/gerar_tabulares_por_secao.py")],check=True)
    subprocess.run([python_documentos,str(ROOT/"scripts/gerar_dossie_tecnico.py")],check=True)
    display(pd.DataFrame([{"arquivo":str(p.relative_to(ROOT)),"bytes":p.stat().st_size} for p in sorted((ROOT/"secoes").glob("*/tabelas/docx/*.docx"))]))
print(rel.report(CTX))
print("Matriz:",ROOT/"outputs/relatorios/Matriz_de_Cobertura_do_Capitulo.md")


# 7 Pacote de resultados e exportação opcional ao Drive

<details>
<summary>Descrição e método</summary>

Ative SALVAR_RESULTADOS_NO_DRIVE para uma execução pessoal no Colab. A célula cria uma pasta datada com a hierarquia preservada; autenticação é pessoal, sem tokens no notebook. O pacote contém todas as saídas locais.

</details>

In [ ]:
ZIP_RESULTADOS=editorial.zip_results(CTX)
print("Resultados compactados:",ZIP_RESULTADOS)
try:
    IS_COLAB=importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:IS_COLAB=False
if SALVAR_RESULTADOS_NO_DRIVE and IS_COLAB:
    display(fv.sync_outputs_drive(CTX,PASTA_RESULTADOS_ID))
elif SALVAR_RESULTADOS_NO_DRIVE:
    print("Exportação configurada para Colab; arquivos locais prontos.")
else:
    print("Saídas locais concluídas. Exportação pessoal ao Drive desativada por padrão.")
